In [9]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.utils import io
import hashlib

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import Dataset, DataLoader

models_path = os.path.abspath(os.path.join('..', 'models'))
sys.path.append(models_path)
models_path = os.path.abspath(os.path.join('..', 'src'))
sys.path.append(models_path)
# from data_preprocessing import save_assistments_DKT
# save_assistments_DKT()


In [4]:
with open('../data/preprocessed/assistments_user_dict.json', 'r') as json_file:
    user_dict = json.load(json_file)

The numbe of unique exercises is too high, we need random vector representations.
We should transform to the following dimension:

In [17]:
round(np.log(dim))

10

In [11]:
# Transform to deterministic random vector function
def transform_to_random_vector(problem_id, correct, dim=10):
    unique_seed = f"{problem_id}_{correct}"
    seed = int(hashlib.md5(unique_seed.encode()).hexdigest(), 16) % (10**8)
    np.random.seed(seed)
    random_vector = np.random.randn(dim)
    return random_vector

# Custom embedding layer to handle (problem_id, correct) pairs
class CustomEmbeddingLayer(nn.Module):
    def __init__(self, dim=10):
        super(CustomEmbeddingLayer, self).__init__()
        self.dim = dim

    def forward(self, input_pairs):
        """
        Input:
            input_pairs (Tensor): shape (batch_size, seq_len, 2), each entry is (problem_id, correct)
        Output:
            Tensor of shape (batch_size, seq_len, dim) with embeddings
        """
        batch_size, seq_len, _ = input_pairs.size()
        embedded = torch.zeros(batch_size, seq_len, self.dim, dtype=torch.float32)
        
        for i in range(batch_size):
            for j in range(seq_len):
                problem_id, correct = input_pairs[i, j]
                problem_id = int(problem_id.item())
                correct = int(correct.item())
                random_vector = transform_to_random_vector(problem_id, correct, self.dim)
                embedded[i, j] = torch.tensor(random_vector, dtype=torch.float32)
        
        return embedded

# Define the DKT model
class DKT(nn.Module):
    def __init__(self, num_items, embed_dim, hid_size, num_hid_layers, drop_prob):
        super(DKT, self).__init__()
        
        # Custom embedding layer for (problem_id, correct) pairs
        self.embedding = CustomEmbeddingLayer(embed_dim)
        
        # RNN layer
        self.rnn = nn.RNN(embed_dim, hid_size, num_hid_layers, batch_first=True)
        
        # Dropout layer for regularization
        self.dropout = nn.Dropout(p=drop_prob)
        
        # Output layer mapping hidden states to probabilities
        self.out = nn.Linear(hid_size, num_items)
        
        # Sigmoid for probability output
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, inputs, lengths):
        # Embed the input sequence
        embedded = self.embedding(inputs)
        
        # Pack the padded sequence for the RNN
        packed_embedded = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)

        # RNN processing
        packed_output, _ = self.rnn(packed_embedded)

        # Unpack the sequence
        output, _ = pad_packed_sequence(packed_output, batch_first=True)

        # Apply dropout
        output = self.dropout(output)

        # Output layer for probabilities
        output = self.out(output)

        # Sigmoid to convert logits to probabilities
        output = self.sigmoid(output)

        mask = torch.arange(output.size(1)).expand(len(lengths), output.size(1)) < lengths.unsqueeze(1)

        # Expand mask for the number of items and apply it to the output
        mask = mask.unsqueeze(-1).expand_as(output)  # Shape: (batch_size, max_seq_length, num_items)
        masked_output = output * mask  # Zero out the padded positions

        return masked_output # Adjust shape for loss function compatibility


In [12]:
# Sample data for three students with varying sequence lengths
input_pairs = [
    [[123, 1], [143, 0], [200, 1]],     # Student 1's answer history
    [[215, 1], [220, 0]],               # Student 2's answer history
    [[99, 0], [85, 1], [101, 1], [102, 0]] # Student 3's answer history
]

# Convert input_pairs to a tensor and pad to maximum sequence length (4 in this case)
max_len = max(len(seq) for seq in input_pairs)
padded_input_pairs = [seq + [[0, 0]] * (max_len - len(seq)) for seq in input_pairs]
input_tensor = torch.tensor(padded_input_pairs)

# Sequence lengths (for packing)
lengths = torch.tensor([len(seq) for seq in input_pairs])


In [138]:
class AnswerSet(Dataset):
    def __init__(self, user_dict):

      user_dict = {key: value for key, value in user_dict.items() if len(value) > 1}

      self.user_ids = list(user_dict.keys())

      # storing a list of the answers
      user_sequences = user_dict.values()
      self.inputs = [user_sequence[:-1] for user_sequence in user_sequences]

      # storing a list of their labels
      self.targets = [[answer[1] for answer in user_sequence[1:]] for user_sequence in user_sequences]

    def __getitem__(self, indices):
      return self.inputs[indices], self.targets[indices]

    def __len__(self):
        return len(self.inputs)

In [142]:
# Define collate function
def collate_batch(batch):
    # Unpacking batches into sequences and labels
    sequences, labels = zip(*batch)  # This will now work as expected

    # Convert sequences to padded tensors
    sequences_tensor = pad_sequence([torch.tensor(seq, dtype=torch.long) for seq in sequences], 
                                    batch_first=True, padding_value=0)

    # Pad the labels similarly
    # Convert labels to padded tensors
    labels_tensor = pad_sequence([torch.tensor(label, dtype=torch.float) for label in labels], 
                                 batch_first=True, padding_value=0)  # or another padding value if needed

    # Create lengths tensor for sequences (before padding)
    lengths_tensor = torch.tensor([len(seq) for seq in sequences])

    return sequences_tensor, labels_tensor, lengths_tensor

# Now create your dataset and dataloader
dataset = AnswerSet(user_dict)
loader = DataLoader(dataset, batch_size=100, collate_fn=collate_batch, num_workers=0)  # Set num_workers to 0 for testing
# just testing the output
sequences, labels, lengths = next(iter(loader))
print(f"Sequences shape: {sequences.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Lengths: {lengths}")


Sequences shape: torch.Size([100, 820, 2])
Labels shape: torch.Size([100, 820])
Lengths: tensor([ 24,  21,   5,   2, 303,  23,   9, 615,   9,   9,  31,   9, 116,   1,
         20, 290,   5,  47, 276, 117, 187, 154, 113, 194,  25,  84, 314, 259,
        235, 467, 325, 152, 651, 514,  78, 721, 144, 680, 515, 363, 410, 680,
         92, 312, 384,  81, 366, 381, 820, 128,  28,  88,  34, 482,  88, 178,
        508, 271, 476,  60,  55,   9,   1,  65,   2,  14,   9,  13,  20,   9,
         12,   9,   7,  31,  38, 152,   9,  13,  33,   2,  17,  10,  35,  49,
          7,  32,  20,  32,  21,  14,  48,  33,  45,  19,   9,   9,   9,  20,
         93,  53])


In [144]:
# Instantiate the model
num_items = 300  # Total number of different questions
embed_dim = 10   # Embedding dimension from transform_to_random_vector
hid_size = 200    # Hidden layer size in the RNN
num_hid_layers = 2  # Number of hidden layers in the RNN
drop_prob = 0.5     # Dropout probability

model = DKT(num_items=num_items, embed_dim=embed_dim, hid_size=hid_size, num_hid_layers=num_hid_layers, drop_prob=drop_prob)

# Forward pass
output = model(sequences, lengths)
print(output)


tensor([[[0.5269, 0.5009, 0.5097,  ..., 0.4958, 0.4736, 0.4748],
         [0.5370, 0.5099, 0.5274,  ..., 0.5042, 0.5277, 0.5408],
         [0.5216, 0.5034, 0.4924,  ..., 0.4916, 0.5165, 0.5033],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

        [[0.5353, 0.4865, 0.5322,  ..., 0.5474, 0.4928, 0.4774],
         [0.5237, 0.4808, 0.4985,  ..., 0.5410, 0.4813, 0.5274],
         [0.5169, 0.5156, 0.5459,  ..., 0.4980, 0.5350, 0.4727],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

        [[0.5140, 0.4957, 0.5253,  ..., 0.5471, 0.5295, 0.4740],
         [0.5339, 0.5249, 0.5109,  ..., 0.5315, 0.5060, 0.4813],
         [0.5790, 0.5351, 0.5099,  ..., 0.4875, 0.4889, 0.

In [ ]:
loss = F.binary_cross_entropy(masked_output, target, reduction='none')
loss = loss * mask  # Zero out the padded positions in the loss
loss = loss.sum() / mask.sum()  # Average only over valid positions

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression


df = pd.read_csv('round_1_train.csv')
model = LinearRegression()
model.fit(df[['feature1']], df['target'], )
print(f'coefficient: {model.coef_}')
print(f'intercept: {model.intercept_}')
      